Continuing with the example from the previous chapter, here is how we would train a sequence classifier on one batch:

from torch.optim import AdamW ব্যবহার করা হয় PyTorch-এ একটি অত্যন্ত শক্তিশালী এবং আধুনিক অপটিমাইজার (Optimizer) ব্যবহার করার জন্য। এটি মূলত ট্রান্সফরমার (Transformers), BERT, GPT বা বড় ধরণের ডিপ লার্নিং মডেলগুলোকে দ্রুত এবং ভালোভাবে প্রশিক্ষণ (Train) দেওয়ার জন্য ব্যবহৃত হয়।

কেন AdamW ব্যবহার করবেন? (প্রধান কারণসমূহ):

১. মেট্রিক বা ওয়েট ডিকাপলিং (Decoupled Weight Decay):
সাধারণ Adam অপটিমাইজারের সাথে weight decay (নিয়মিতকরণ) ঠিকঠাক কাজ করে না, যা মডেলের পারফরম্যান্স কমিয়ে দেয়। AdamW (Adam with Weight Decay) এই সমস্যাটি সমাধান করে, সরাসরি মডেলের প্যারামিটারে Weight Decay প্রয়োগ করে।

২. ভালো জেনারেলাইজেশন (Better Generalization):
AdamW সাধারণ Adam-এর চেয়ে ভালো কাজ করে কারণ এটি মডেলকে শুধু ট্রেইনিং ডেটা মুখস্থ না করে, নতুন ডেটাতেও ভালো ফলাফল (Generalize) করতে সাহায্য করে।

৩. ট্রান্সফরমার মডেলের জন্য আদর্শ:
Hugging Face বা যেকোনো ট্রান্সফরমার ভিত্তিক মডেলে এখন AdamW ব্যবহার করাই স্ট্যান্ডার্ড নিয়ম।

৪. Transformer লাইব্রেরির পরিবর্তে:
আগে অনেকেই from transformers import AdamW ব্যবহার করতেন, কিন্তু এখন সেটি transformers লাইব্রেরি থেকে সরিয়ে নেওয়া হয়েছে। এখন সরাসরি PyTorch-এর torch.optim থেকে এটি ব্যবহার করতে বলা হয়।

In [3]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequences = [
    "I am sakhawat hossen",
    "I am from ctg",
    "I love Machine Learning"
]

batch = tokenizer(sequences,padding=True,truncation=True,return_tensors="pt")

batch

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'input_ids': tensor([[  101,  1045,  2572,  7842, 15256, 24281,  7570, 14416,   102],
        [  101,  1045,  2572,  2013, 14931,  2290,   102,     0,     0],
        [  101,  1045,  2293,  3698,  4083,   102,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0]])}

তুমি একটা BERT মডেল লোড করছ (bert-base-uncased) যেটা সাধারণত Sequence Classification (যেমন: পজিটিভ/নেগেটিভ, স্প্যাম/নন-স্প্যাম) এর জন্য ব্যবহার হয়।
তারপর ৩টা ছোট টেক্সটকে টোকেনাইজ করে একটা batch তৈরি করছ, এবং শেষে দুটো লেবেল (1, 1) যোগ করছ।

এর মানে কী?

* bert-base-uncased চেকপয়েন্টটা আসলে Masked Language Modeling (MLM) + Next Sentence Prediction (NSP) এর জন্য প্রি-ট্রেইন করা (BERT-এর অরিজিনাল টাস্ক)।
* কিন্তু তুমি লোড করছ AutoModelForSequenceClassification → এটার আর্কিটেকচারে অতিরিক্ত একটা classifier লেয়ার থাকে (দুইটা লিনিয়ার লেয়ার: classifier.weight + bias)।
* চেকপয়েন্টে classifier লেয়ার নেই → তাই MISSING বলছে। Hugging Face অটোমেটিক নতুন করে এই লেয়ারগুলো র‍্যান্ডমলি ইনিশিয়ালাইজ করে দিচ্ছে।
* অন্যদিকে, MLM/NSP-এর জন্য যে লেয়ার আছে (cls.predictions..., cls.seq_relationship...) সেগুলো Sequence Classification-এ দরকার নেই → তাই UNEXPECTED বলছে।

{
  
  'input_ids': tensor([

    [ 101, 1045, 2572, 7842,15256,24281, 7570,14416,  102],   # "I am sakhawat hossen"
    [ 101, 1045, 2572, 2013,14931, 2290,  102,    0,    0],   # "I am from ctg" (প্যাডিং 0)
    [ 101, 1045, 2293, 3698, 4083,  102,    0,    0,    0]    # "I love Machine Learning" (প্যাডিং 0)
  ]),
  
  'token_type_ids': tensor([[0]*9, [0]*9, [0]*9]),             # সব 0 (single sentence)
  
  'attention_mask': tensor([

    [1,1,1,1,1,1,1,1,1],   # প্রথমটা ৯টা টোকেন
    [1,1,1,1,1,1,1,0,0],   # দ্বিতীয়টা ৭টা + ২টা প্যাড
    [1,1,1,1,1,1,0,0,0]    # তৃতীয়টা ৬টা + ৩টা প্যাড
  ])
}

ব্যাখ্যা:

101 = [CLS] টোকেন (শুরু)

102 = [SEP] টোকেন (শেষ)

0 = প্যাডিং টোকেন (padding=True করেছ বলে সবাইকে একই লম্বা করা হয়েছে)

attention_mask দিয়ে মডেলকে বলা হয়েছে কোন টোকেন আসল, কোনটা প্যাডিং (প্যাডিং-কে ইগনোর করতে)

In [4]:
batch["labels"] = torch.tensor([1,1,1])

এখন batch-এ নতুন কী যোগ হলো:
'labels': tensor([1, 1,1])
মানে কী?

তুমি বলে দিচ্ছ যে সেন্টেন্সের লেবেল = 1 (যেমন: পজিটিভ, class 1)

In [5]:
batch

{'input_ids': tensor([[  101,  1045,  2572,  7842, 15256, 24281,  7570, 14416,   102],
        [  101,  1045,  2572,  2013, 14931,  2290,   102,     0,     0],
        [  101,  1045,  2293,  3698,  4083,   102,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0]]), 'labels': tensor([1, 1, 1])}

In [6]:
optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

In [7]:
loss


tensor(0.6521, grad_fn=<NllLossBackward0>)

১. optimizer = AdamW(model.parameters())
কী হচ্ছে?

AdamW হলো একটা অপটিমাইজার (Optimizer) – এটা মডেলের ওজন (weights) কীভাবে আপডেট হবে তা ঠিক করে।
model.parameters() → মডেলের সব লার্নেবল প্যারামিটার (weight + bias) এর লিস্ট দিচ্ছে।
AdamW হলো Adam অপটিমাইজারের একটা উন্নত ভার্সন (weight decay ভালোভাবে হ্যান্ডেল করে) – BERT/Transformer ফাইন-টিউনিং-এ সবচেয়ে বেশি ব্যবহার হয়।

সহজ কথায়:
তুমি মডেলকে বলছ – “এই সব ওজনগুলোকে আমি তোমার হাতে দিলাম, তুমি এগুলোকে ধীরে ধীরে ঠিক করো যাতে loss কমে যায়।”
নোট: সাধারণত এখানে learning rateও দেওয়া হয়, যেমন:
Pythonoptimizer = AdamW(model.parameters(), lr=2e-5)   # lr = learning rate, ছোট রাখা হয় BERT-এর জন্য
২. loss = model(**batch).loss
কী হচ্ছে?

model(**batch) → মডেলকে পুরো batch (input_ids, attention_mask, labels সব) দিয়ে ফরওয়ার্ড পাস করা হচ্ছে।
BERT SequenceClassification মডেল logits বের করে (প্রত্যেক ক্লাসের স্কোর)।
তারপর মডেল নিজেই cross-entropy loss ক্যালকুলেট করে (কারণ labels দেওয়া আছে)।
.loss → সেই ক্যালকুলেটেড লস ভ্যালু (একটা scalar টেনসর)।

সহজ কথায়:
মডেলকে ডেটা দিয়ে বলা হলো – “এই টেক্সটগুলো দেখে প্রেডিকশন করো, তারপর বলো তুমি কতটা ভুল করেছ (loss)।”
loss যত বড়, মডেল তত খারাপ পারফর্ম করছে।
উদাহরণ আউটপুট (প্রথমবার চালালে):
Loss ≈ 0.6 থেকে 1.5 এর মধ্যে থাকে (র‍্যান্ডম classifier-এর জন্য)।
৩. loss.backward()
কী হচ্ছে?

এটাই backpropagation।
loss-এর উপর ভিত্তি করে মডেলের সব প্যারামিটারের gradient (ঢাল) ক্যালকুলেট করে।
gradient বলে দেয়: “যদি এই ওজনকে একটু বাড়াই/কমাই, তাহলে loss কতটা কমবে বা বাড়বে?”

সহজ কথায়:
মডেলকে জিজ্ঞাসা করা হলো – “তুমি যেখানে ভুল করেছ, সেখানে কোন দিকে ঠিক করতে হবে? কতটা ঠিক করতে হবে?”
এখন প্রত্যেক ওজনের পাশে একটা gradient সংরক্ষিত হয়ে গেল।
৪. optimizer.step()
কী হচ্ছে?

অপটিমাইজার (AdamW) এখন সব gradient দেখে মডেলের ওজনগুলোকে আপডেট করে।
ফর্মুলা:
new_weight = old_weight - (learning_rate × gradient)
(AdamW আরও স্মার্ট ভাবে করে, momentum + adaptive LR সহ)

সহজ কথায়:
“ঠিক আছে, তুমি যেদিকে ঠিক করতে বলেছ, সেই দিকে একটু একটু করে এগিয়ে যাও।”
এই এক লাইন চালানোর পর মডেলের ওজন একটু ভালো হয়ে যায় (loss কমার দিকে)।

# Loading a dataset from the Hub

এখানে আমরা Hugging Face Hub থেকে ডেটাসেট লোড করার প্রক্রিয়া দেখবো, বিশেষ করে MRPC ডেটাসেট (GLUE বেঞ্চমার্কের একটা অংশ)

### আমরা কোন ডেটাসেট নিচ্ছি?
MRPC (Microsoft Research Paraphrase Corpus)

এটা GLUE বেঞ্চমার্কের একটা টাস্ক।
কাজ: দুটো বাক্য দেওয়া হবে → বলতে হবে দুটো কি একই অর্থের (equivalent) না আলাদা অর্থের (not equivalent)।
লেবেল: 0 = not equivalent, 1 = equivalent



In [8]:
from datasets import load_dataset

raw_datasets = load_dataset("glue","mrpc")

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

* "glue" হলো ডেটাসেটের গ্রুপের নাম (GLUE বেঞ্চমার্ক)।
* "mrpc" হলো সেই গ্রুপের ভিতরের নির্দিষ্ট ডেটাসেট।

In [9]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

* raw_datasets হলো একটা DatasetDict (ডিকশনারির মতো)।
* এর ভিতর ৩টা স্প্লিট আছে: train, validation, test।
* train-এ ৩৬৬৮টা উদাহরণ (পেয়ার অফ সেন্টেন্স)।
* validation-এ ৪০৮টা, test-এ ১৭২৫টা।
* উদাহরণে ৪টা কলাম: sentence1, sentence2, label, idx।

## ট্রেনিং ডেটা

In [10]:
raw_train_dataset = raw_datasets["train"]

In [11]:
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

* raw_datasets["train"] → শুধু ট্রেনিং অংশটা নেওয়া হলো।
* raw_train_dataset[0] → প্রথম উদাহরণ দেখানো হলো।
* label = 1 মানে দুটো বাক্য equivalent (একই অর্থ বোঝাচ্ছে, শুধু শব্দের ক্রম/ভাষা একটু আলাদা)।

## features

In [12]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

* label হলো ClassLabel টাইপ।
* এর names লিস্টে দুটো নাম আছে:

    * 0 → not_equivalent (অর্থ আলাদা)

    * 1 → equivalent (অর্থ একই)

* এজন্য label যখন 1 দেখছো, মানে দুটো বাক্য একই অর্থ বোঝাচ্ছে।

# Preprocessing a dataset


## ১. কেন প্রিপ্রসেসিং দরকার?
মডেল শুধু টেক্সট বোঝে না — এটা শুধু নাম্বার (টোকেন আইডি) নিয়ে কাজ করে।
তাই দরকার:

টেক্সট → টোকেন আইডি
দুটো বাক্যকে একসাথে জোড়া করে ইনপুট তৈরি ([CLS] sentence1 [SEP] sentence2 [SEP])
token_type_ids যোগ করা (প্রথম বাক্য = 0, দ্বিতীয় = 1)
attention_mask যোগ করা
padding & truncation (লম্বা/ছোট বাক্যকে একই লম্বা করা)

## ২. সিম্পল উপায়ে টোকেনাইজ করা

In [17]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized_sent = tokenizer(raw_datasets["train"]["sentence1"][1])
tokenized_sent


{'input_ids': [101, 9805, 3540, 11514, 2050, 3079, 11282, 2243, 1005, 1055, 2077, 4855, 1996, 4677, 2000, 3647, 4576, 1999, 2687, 2005, 1002, 1016, 1012, 1019, 4551, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

### Decode the token

In [18]:
tokenizer.convert_ids_to_tokens(tokenized_sent["input_ids"])

['[CLS]',
 'yu',
 '##ca',
 '##ip',
 '##a',
 'owned',
 'dominic',
 '##k',
 "'",
 's',
 'before',
 'selling',
 'the',
 'chain',
 'to',
 'safe',
 '##way',
 'in',
 '1998',
 'for',
 '$',
 '2',
 '.',
 '5',
 'billion',
 '.',
 '[SEP]']

# ৩. পুরো ডেটাসেট টোকেনাইজ করার সিম্পল (কিন্তু অদক্ষ) উপায়

#### we will use the Dataset.map() method. This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset, so let’s define a function that tokenizes our inputs:

মূল কথা: আমরা কেন Dataset.map() ব্যবহার করব?
আগের সিম্পল উপায়ে tokenizer চালালে:

পুরো ডেটা মেমোরিতে লোড হয়ে যায় → RAM বেশি লাগে
আউটপুট শুধু dictionary হয় → Datasets-এর সুবিধা হারায়

তাই আমরা Datasets library-এর map() ব্যবহার করব।
map() মানে: “পুরো ডেটাসেটের প্রত্যেকটা উদাহরণে (example) একটা ফাংশন চালাও, আর নতুন ফিল্ড যোগ করো।”

In [23]:
def tokenized_function(example):
  return tokenizer(example["sentence1"] , example["sentence2"] , truncation=True)

এই ফাংশন কী করে?

* input: একটা example (dictionary) → যেমন {'sentence1': "...", 'sentence2': "...", 'label': 1}
* output: tokenizer থেকে পাওয়া নতুন dictionary → {'input_ids': [...], 'attention_mask': [...], 'token_type_ids': [...]}

বিশেষ কথা:

* tokenizer lists নিতে পারে → তাই যদি batched=True হয়, তাহলে example["sentence1"] হবে একটা list of sentences
* truncation=True → লম্বা sentence কেটে ফেলে (max_length-এর বাইরে গেলে)
* padding এখানে করা হয়নি → কেন? পরে বুঝাবো

### map() দিয়ে পুরো ডেটাসেটে চালানো

In [24]:
tokenized_datasets = raw_datasets.map(tokenized_function,batched = True)


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

batched=True কেন দিলাম?

* Normal map() → একটা একটা করে example নেয় → ধীরগতি
* batched=True → একসাথে ১০০০/২০০০টা example নেয় → tokenizer খুব দ্রুত কাজ করে (Rust backend-এর জন্য)
* ফলে preprocessing ৫-১০ গুণ দ্রুত হয়!

In [25]:
tokenized_datasets


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

padding=True দিয়ে map() করলে → পুরো ডেটাসেটের সবচেয়ে লম্বা sentence-এর length পর্যন্ত সবাইকে pad করবে → অনেক জায়গায় 0 (padding) বসবে → মেমোরি ও সময় নষ্ট

### আরও দ্রুত করার উপায়

যদি তোমার tokenizer fast না হয় (Rust-based না) → map()-এ num_proc=4 দিতে পারো (multiprocessing)

In [26]:
tokenized_datasets = raw_datasets.map(tokenized_function,batched = True,num_proc=4)

Map (num_proc=4):   0%|          | 0/3668 [00:00<?, ? examples/s]

# **Dynamic padding**

কেন Dynamic Padding দরকার?

আমাদের tokenized_datasets-এ প্রত্যেক sentence-এর length আলাদা আলাদা (যেমন ৩২, ৫০, ৬৭ ইত্যাদি)।
যদি আমরা সবগুলোকে একই length-এ প্যাড করি (যেমন পুরো ডেটাসেটের সবচেয়ে লম্বা ২৫০ পর্যন্ত), তাহলে:

প্রত্যেক input-এ অনেক 0 (padding token) বসবে → waste
Batch-এ প্রত্যেক টাইম GPU/TPU-তে বেশি ক্যালকুলেশন → ট্রেনিং slow
মেমোরি বেশি লাগবে

তাই আমরা dynamic padding করব:
প্রত্যেক batch-এর ভিতরে শুধু সেই batch-এর longest sequence পর্যন্ত প্যাড করবো।
যেমন: একটা batch-এ max length ৬৭ হলে সবাইকে ৬৭ পর্যন্ত প্যাড, পরের batch-এ ৮৫ হলে সেটা ৮৫ পর্যন্ত।
এতে:

Training অনেক দ্রুত হয়
কম মেমোরি লাগে
কিন্তু TPU-তে সমস্যা হতে পারে (TPU fixed shape পছন্দ করে)

## কীভাবে Dynamic Padding করা হয়? → DataCollatorWithPadding

এই data_collator কী করে?

* tokenizer জানে কোন padding_token ব্যবহার করতে হবে (BERT-এর জন্য 0)
* padding কোন দিকে করতে হবে (left বা right – BERT-এ right)
* প্রত্যেক batch-এর samples নিয়ে অটোমেটিক সবচেয়ে লম্বা length পর্যন্ত প্যাড করে দেয়
* input_ids, attention_mask, token_type_ids সব ঠিক করে দেয়

In [28]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [29]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}

In [30]:
samples

{'label': [1, 0, 1, 0, 1, 1, 0, 1],
 'input_ids': [[101,
   2572,
   3217,
   5831,
   5496,
   2010,
   2567,
   1010,
   3183,
   2002,
   2170,
   1000,
   1996,
   7409,
   1000,
   1010,
   1997,
   9969,
   4487,
   23809,
   3436,
   2010,
   3350,
   1012,
   102,
   7727,
   2000,
   2032,
   2004,
   2069,
   1000,
   1996,
   7409,
   1000,
   1010,
   2572,
   3217,
   5831,
   5496,
   2010,
   2567,
   1997,
   9969,
   4487,
   23809,
   3436,
   2010,
   3350,
   1012,
   102],
  [101,
   9805,
   3540,
   11514,
   2050,
   3079,
   11282,
   2243,
   1005,
   1055,
   2077,
   4855,
   1996,
   4677,
   2000,
   3647,
   4576,
   1999,
   2687,
   2005,
   1002,
   1016,
   1012,
   1019,
   4551,
   1012,
   102,
   9805,
   3540,
   11514,
   2050,
   4149,
   11282,
   2243,
   1005,
   1055,
   1999,
   2786,
   2005,
   1002,
   6353,
   2509,
   2454,
   1998,
   2853,
   2009,
   2000,
   3647,
   4576,
   2005,
   1002,
   1015,
   1012,
   1022,
   4551,
   1

In [31]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

Dynamic padding + DataCollatorWithPadding = modern best practice
এটা ব্যবহার করলে ট্রেনিং ২-৩ গুণ দ্রুত হতে পারে, বিশেষ করে variable length ডেটাতে (যেমন MRPC)।